In [1]:
import sys
!{sys.executable} -m pip install youtube-transcript-api spacy networkx -q

import os

# Define o diretório raiz para salvar as transcrições localmente
BASE_DIR = '../data/raw'
os.makedirs(BASE_DIR, exist_ok=True)

print(f"\n[OK] Diretório de destino configurado: {BASE_DIR}")


[OK] Diretório de destino configurado: ../data/raw


In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
import re
import json
import os

def extract_video_id(url):
    """Extrai o ID do vídeo a partir de diferentes formatos de URL do YouTube."""
    regex = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
    match = re.search(regex, url)
    return match.group(1) if match else None

def get_transcript(video_id, languages=['pt', 'en']):
    """
    Obtém a transcrição do vídeo utilizando a nova sintaxe da API.
    A lista 'languages' define a ordem de prioridade dos idiomas.
    """
    try:
        api = YouTubeTranscriptApi()
        transcript_obj = api.fetch(video_id, languages=languages)
        return transcript_obj.to_raw_data()
    except Exception as e:
        print(f"  [ERRO] Não foi possível obter transcrição para o ID {video_id}. Detalhes: {e}")
        return None

def save_transcript(video_id, transcript, base_dir):
    """Salva a transcrição nos formatos JSON e TXT."""
    if not transcript:
        return False

    # 1. Salva como JSON (preserva os timestamps, útil para os metadados do seu grafo NER depois)
    json_path = os.path.join(base_dir, f"{video_id}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(transcript, f, ensure_ascii=False, indent=4)

    # 2. Salva como TXT puro (texto contínuo para processamento NLP)
    # Correção: Extrai diretamente a chave 'text' de cada dicionário da lista
    text_formatted = "\n".join([item['text'] for item in transcript])

    text_path = os.path.join(base_dir, f"{video_id}.txt")
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(text_formatted)

    print(f"  [SUCESSO] Arquivos {video_id}.json e {video_id}.txt salvos no Drive.")
    return True

In [3]:
# Lista de URLs dos vídeos do YouTube
video_urls = [
    "https://youtu.be/7xTGNNLPyMI?si=Vpw7fdd65-fGPnLF"
]

print(f"Iniciando processamento de {len(video_urls)} vídeo(s)...\n")

for url in video_urls:
    video_id = extract_video_id(url)

    if not video_id:
        print(f"[AVISO] URL inválida ignorada: {url}")
        continue

    # Regra de cache: verifica se o processamento já foi feito antes
    expected_txt_file = os.path.join(BASE_DIR, f"{video_id}.txt")
    if os.path.exists(expected_txt_file):
        print(f"[-] Vídeo ID {video_id} já processado anteriormente. Pulando...")
        continue

    print(f"[+] Processando vídeo ID: {video_id}...")
    transcript_data = get_transcript(video_id, languages=['pt', 'pt-BR', 'en'])

    if transcript_data:
        save_transcript(video_id, transcript_data, BASE_DIR)

print("\nProcessamento finalizado!")

Iniciando processamento de 1 vídeo(s)...

[+] Processando vídeo ID: 7xTGNNLPyMI...
  [SUCESSO] Arquivos 7xTGNNLPyMI.json e 7xTGNNLPyMI.txt salvos no Drive.

Processamento finalizado!


In [4]:
import sys
!{sys.executable} -m spacy download pt_core_news_lg

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 5.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')


In [5]:
import sys
!{sys.executable} -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
